In [ ]:
# ============================================================
# NOTEBOOK 1 - PARTE 1
# ORBITAL INSPECTION
# ============================================================

import os
import re
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.timeseries import BoxLeastSquares

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = (
    "/home/luciam/Documents/starry/"
    "Orion_yceb/dat_orion_sergio/sample_3/"
    "831359(1)/Light_Curve_Files/"
)

# ============================================================
# FIND FILES
# ============================================================

files = sorted(
    glob.glob(
        os.path.join(
            BASE_PATH,
            "*.txt"
        )
    )
)

print(
    f"Found {len(files)} files"
)

# ============================================================
# GROUP BY TIC
# ============================================================

tic_files = {}

for f in files:

    fname = os.path.basename(f)

    match = re.search(
        r'tic\s+(\d+)_S_(\d+)_LC',
        fname
    )

    if match is None:
        continue

    tic = int(
        match.group(1)
    )

    sector = int(
        match.group(2)
    )

    tic_files.setdefault(
        tic,
        []
    )

    tic_files[tic].append(
        (sector, f)
    )

print(
    f"Found {len(tic_files)} TICs"
)

# ============================================================
# LOAD LIGHT CURVE
# ============================================================

def load_sector_file(fname):

    df = pd.read_csv(
        fname,
        delim_whitespace=True,
        comment="#",
        header=None
    )

    arr = df.values

    time_col = None
    flux_col = None

    for i in range(arr.shape[1]):

        med = np.nanmedian(
            arr[:, i]
        )

        if (
            time_col is None
            and
            1000 < med < 5000
        ):
            time_col = i

        if (
            flux_col is None
            and
            0.5 < med < 1.5
        ):
            flux_col = i

    if time_col is None:
        raise ValueError(
            f"TIME column not found in {fname}"
        )

    if flux_col is None:
        raise ValueError(
            f"FLUX column not found in {fname}"
        )

    time = arr[:, time_col]
    flux = arr[:, flux_col]

    mask = (
        np.isfinite(time)
        &
        np.isfinite(flux)
    )

    time = time[mask]
    flux = flux[mask]

    srt = np.argsort(time)

    return (
        time[srt],
        flux[srt]
    )

# ============================================================
# BUILD GLOBAL LC
# ============================================================

def build_global_lc(sectors):

    all_time = []
    all_flux = []

    for sector, fname in sectors:

        t, f = load_sector_file(
            fname
        )

        f = (
            f /
            np.nanmedian(f)
        )

        all_time.append(t)
        all_flux.append(f)

    time = np.concatenate(
        all_time
    )

    flux = np.concatenate(
        all_flux
    )

    srt = np.argsort(time)

    return (
        time[srt],
        flux[srt]
    )

# ============================================================
# BLS
# ============================================================

def get_bls_period(
    time,
    flux,
    min_period=0.5,
    max_period=20
):

    periods = np.linspace(
        min_period,
        max_period,
        20000
    )

    durations = np.linspace(
        0.02,
        0.20,
        20
    )

    bls = BoxLeastSquares(
        time,
        flux
    )

    result = bls.power(
        periods,
        durations
    )

    best = np.argmax(
        result.power
    )

    return (
        result.period[best],
        result
    )

# ============================================================
# BINNING
# ============================================================

def phase_bin_curve(
    phase,
    flux,
    nbins=120
):

    bins = np.linspace(
        0,
        1,
        nbins + 1
    )

    centers = 0.5 * (
        bins[:-1]
        + bins[1:]
    )

    medians = np.full(
        nbins,
        np.nan
    )

    for i in range(nbins):

        m = (
            (phase >= bins[i])
            &
            (phase < bins[i+1])
        )

        if np.sum(m) > 5:

            medians[i] = np.nanmedian(
                flux[m]
            )

    return (
        centers,
        medians
    )

# ============================================================
# PHASE PANEL
# ============================================================

def phase_panel(
    ax,
    time,
    flux,
    period,
    title
):

    phase = (
        (time % period)
        / period
    )

    centers, medians = (
        phase_bin_curve(
            phase,
            flux,
            nbins=80
        )
    )

    ax.scatter(
        phase,
        flux,
        s=1,
        alpha=0.02
    )

    ax.plot(
        centers,
        medians,
        lw=3
    )

    ax.set_xlim(
        0,
        1
    )

    ax.set_title(
        title
    )

# ============================================================
# PERIOD DIAGNOSTIC
# ============================================================

def inspect_periods(tic):

    sectors = sorted(
        tic_files[tic]
    )

    time_all, flux_all = (
        build_global_lc(
            sectors
        )
    )

    P_bls, result = (
        get_bls_period(
            time_all,
            flux_all
        )
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15,4)
    )

    phase_panel(
        axes[0],
        time_all,
        flux_all,
        P_bls/2,
        f"P/2 = {P_bls/2:.4f}"
    )

    phase_panel(
        axes[1],
        time_all,
        flux_all,
        P_bls,
        f"P = {P_bls:.4f}"
    )

    phase_panel(
        axes[2],
        time_all,
        flux_all,
        2*P_bls,
        f"2P = {2*P_bls:.4f}"
    )

    plt.suptitle(
        f"TIC {tic}\n"
        f"P_BLS = {P_bls:.5f} d",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

    return P_bls

# ============================================================
# GLOBAL T0
# ============================================================

def get_global_t0(
    tic,
    P_adopted
):

    sectors = sorted(
        tic_files[tic]
    )

    time_all, flux_all = (
        build_global_lc(
            sectors
        )
    )

    phase = (
        (time_all % P_adopted)
        / P_adopted
    )

    centers, medians = (
        phase_bin_curve(
            phase,
            flux_all
        )
    )

    phase_ref = centers[
        np.nanargmin(
            medians
        )
    ]

    t0 = (
        phase_ref
        *
        P_adopted
    )

    return (
        t0,
        phase_ref
    )

In [ ]:
tic = 11300044

P_bls = inspect_periods(tic)

In [ ]:
P_adopted = 2 * P_bls

In [ ]:
t0, phase_ref = get_global_t0(
    tic,
    P_adopted
)

print(t0)
print(phase_ref)

In [ ]:
def inspect_sectors(
    tic,
    P_adopted,
    t0
):

    sectors = sorted(
        tic_files[tic]
    )

    nsec = len(sectors)

    fig, axes = plt.subplots(
        nsec,
        2,
        figsize=(12, 3*nsec)
    )

    if nsec == 1:
        axes = np.array([axes])

    for i, (sector, fname) in enumerate(
        sectors
    ):

        time, flux = load_sector_file(
            fname
        )

        flux = (
            flux /
            np.nanmedian(flux)
        )

        phase = (
            ((time - t0) % P_adopted)
            / P_adopted
        )

        centers, medians = (
            phase_bin_curve(
                phase,
                flux,
                nbins=120
            )
        )

        # -------------------
        # TIME SERIES
        # -------------------

        axes[i,0].scatter(
            time,
            flux,
            s=2,
            alpha=0.6
        )

        axes[i,0].set_title(
            f"Sector {sector}"
        )

        axes[i,0].set_ylabel(
            "Flux"
        )

        # -------------------
        # PHASE CURVE
        # -------------------

        axes[i,1].scatter(
            phase,
            flux,
            s=2,
            alpha=0.08
        )

        axes[i,1].plot(
            centers,
            medians,
            lw=3
        )

        axes[i,1].set_xlim(
            0,
            1
        )

        axes[i,1].set_title(
            f"Sector {sector}"
        )

    axes[-1,0].set_xlabel(
        "BTJD"
    )

    axes[-1,1].set_xlabel(
        "Phase"
    )

    plt.suptitle(
        f"TIC {tic}\n"
        f"P = {P_adopted:.5f} d",
        fontsize=14
    )

    plt.tight_layout()

    plt.show()

In [ ]:
def check_phase_alignment(
    tic,
    P_adopted,
    t0
):

    sectors = sorted(
        tic_files[tic]
    )

    print("\nPhase minima\n")

    for sector, fname in sectors:

        time, flux = load_sector_file(
            fname
        )

        flux = (
            flux /
            np.nanmedian(flux)
        )

        phase = (
            ((time - t0) % P_adopted)
            / P_adopted
        )

        centers, medians = (
            phase_bin_curve(
                phase,
                flux,
                nbins=120
            )
        )

        phase_min = centers[
            np.nanargmin(
                medians
            )
        ]

        print(
            f"S{sector}: "
            f"{phase_min:.4f}"
        )

In [ ]:
results = []

def save_decision(
    tic,
    P_adopted,
    keep_sectors
):

    results.append({

        "tic": tic,

        "P_adopted": P_adopted,

        "keep_sectors":
            ";".join(
                map(
                    str,
                    keep_sectors
                )
            )

    })

    print(
        f"Saved TIC {tic}"
    )

In [ ]:
save_decision(

    tic=11300044,

    P_adopted=2.10180509,

    keep_sectors=[
        6,
        98
    ]

)

In [ ]:
results_df = pd.DataFrame(
    results
)

results_df

In [ ]:
tic = 249074181
P_bls = inspect_periods(
    tic
)


In [ ]:

P_adopted = 0.81494

t0, phase_ref = get_global_t0(
    tic,
    P_adopted
)

inspect_sectors(
    tic,
    P_adopted,
    t0
)

In [ ]:
save_decision(
    tic,
    P_adopted,
    [5,32,98]
)

In [ ]:
#results_df = results_df.drop(index=11)

#results_df = results_df.reset_index(drop=True)

results_df

In [ ]:
def analyze_sector(
    tic,
    sector,
    min_period=0.5,
    max_period=20
):

    # ------------------------------------
    # localizar archivo
    # ------------------------------------

    sectors = dict(
        tic_files[tic]
    )

    fname = sectors[sector]

    print(
        f"\nTIC {tic}"
    )

    print(
        f"Sector {sector}"
    )

    print(
        fname
    )

    # ------------------------------------
    # cargar curva
    # ------------------------------------

    time, flux = load_sector_file(
        fname
    )

    flux = (
        flux /
        np.nanmedian(flux)
    )

    # ------------------------------------
    # BLS
    # ------------------------------------

    periods = np.linspace(
        min_period,
        max_period,
        20000
    )

    durations = np.linspace(
        0.02,
        0.20,
        40
    )

    bls = BoxLeastSquares(
        time,
        flux
    )

    result = bls.power(
        periods,
        durations
    )

    best = np.argmax(
        result.power
    )

    P_best = result.period[
        best
    ]

    print(
        f"\nBest period = "
        f"{P_best:.6f} d"
    )

    # ------------------------------------
    # FIGURE
    # ------------------------------------

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(16,4)
    )

    # ------------------------------------
    # time series
    # ------------------------------------

    axes[0].scatter(
        time,
        flux,
        s=2
    )

    axes[0].set_xlabel(
        "BTJD"
    )

    axes[0].set_ylabel(
        "Normalized Flux"
    )

    axes[0].set_title(
        f"Sector {sector}"
    )

    # ------------------------------------
    # BLS spectrum
    # ------------------------------------

    axes[1].plot(
        result.period,
        result.power,
        lw=1
    )

    axes[1].axvline(
        P_best,
        ls="--"
    )

    axes[1].set_xlabel(
        "Period [d]"
    )

    axes[1].set_ylabel(
        "BLS Power"
    )

    axes[1].set_title(
        f"P = {P_best:.6f}"
    )

    # ------------------------------------
    # folded curve
    # ------------------------------------

    phase = (
        (time % P_best)
        / P_best
    )

    centers, medians = (
        phase_bin_curve(
            phase,
            flux,
            nbins=120
        )
    )

    axes[2].scatter(
        phase,
        flux,
        s=2,
        alpha=0.05
    )

    axes[2].plot(
        centers,
        medians,
        lw=3
    )

    axes[2].set_xlim(
        0,
        1
    )

    axes[2].set_xlabel(
        "Phase"
    )

    axes[2].set_ylabel(
        "Normalized Flux"
    )

    axes[2].set_title(
        f"P = {P_best:.6f}"
    )

    plt.tight_layout()
    plt.show()

    # ------------------------------------
    # top peaks
    # ------------------------------------

    idx = np.argsort(
        result.power
    )[::-1]

    print(
        "\nTop 10 periods\n"
    )

    for i in idx[:10]:

        print(
            f"{result.period[i]:.6f}",
            result.power[i]
        )

    return (
        P_best,
        result
    )

In [ ]:
analyze_sector(
    tic=712930938,
    sector=98
)

In [ ]:
results_df = pd.DataFrame(
    results
)

results_df

In [ ]:
results_df = (
    results_df
    .drop(index=[8,10])
    .reset_index(drop=True)
)

In [ ]:
results_df

In [ ]:
len(results)

In [ ]:
results[-5:]

In [ ]:
from collections import Counter

counter = Counter(
    r["tic"]
    for r in results
)

for tic, n in counter.items():

    if n > 1:

        print(
            tic,
            n
        )

In [ ]:
unique_results = {}

for r in results:

    unique_results[r["tic"]] = r

results = list(
    unique_results.values()
)

In [ ]:
from collections import Counter

counter = Counter(
    r["tic"]
    for r in results
)

for tic, n in counter.items():

    if n > 1:
        print(tic, n)

In [ ]:
results_df = (
    pd.DataFrame(results)
    .sort_values("tic")
    .reset_index(drop=True)
)

results_df

In [ ]:
print("Rows =", len(results_df))
print("Unique TICs =", results_df["tic"].nunique())

In [ ]:
results_df = (
    results_df
    .sort_values("tic")
    .reset_index(drop=True)
)

results_df.to_csv(
    "orbital_adopted_periods.csv",
    index=False
)

print(
    "Saved orbital_adopted_periods.csv"
)

In [ ]:
results_df

In [ ]:
for _, row in results_df.iterrows():

    tic = row["tic"]

    keep = [
        int(x)
        for x in row["keep_sectors"].split(";")
    ]

    available = [
        s for s, f
        in tic_files[tic]
    ]

    missing = (
        set(keep)
        - set(available)
    )

    if len(missing) > 0:

        print(
            tic,
            "missing:",
            missing
        )

In [ ]:
results_df.sort_values(
    "P_adopted"
)

In [ ]:
results_df.to_csv(
    "orbital_adopted_periods_v1.csv",
    index=False
)